===========================================================================
# Evidence Retrieval and Narrative Generation
===========================================================================

This notebook transforms the validated hospital evidence repository into experiemental prompts for downstream communication-style generation. The following notebook (5 - Evaluation) will review which prompt performs the best alongside the final generation of narratives. This notebook will focus on creating standardized prompts for various communication styles.

* Due to computational limits on the number of tokens per chunk, the project shifted from a RAG based project to a most streamlined retrieval -- the old code has been archived.


## Task 1: Imports and Setup
--------------------------------------------------------------------------

In [1]:
# Import necessary libraries

import pandas as pd
import numpy as np
from collections import Counter
import inspect
import json
import textwrap
import warnings
from pathlib import Path
# Import necessary libraries

import inspect
import json
import random
import re
import textwrap
import time
import warnings
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 160)

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)


#### Formatting and Repositories

In [2]:
# Formatting for printing outputs, function repository, and insight repository
line = "--" * 40
double = "==" * 40


def print_outputs(title = None, *outputs):

    line = "--" * 40
    double = "==" * 40

    if title:
        print(f"{double}\n{title}\n{double}")
    else:
        print(line)

    for item in outputs:

        if len(item) == 2:
            subtitle, output = item
            print_line = True
        else:
            subtitle, output, print_line = item

        if subtitle:
            print(f"{subtitle}")

        print(output)

        if print_line:
            print(line)


# Storing insights inside insights repo
insights_repository = pd.DataFrame(columns = [
    "stage",
    "section",
    "experiment",
    "parameter",
    "metric",
    "value",
    "status",
    "decision",
    "insight",
    "notes"
])


def store_insight(
    insight,
    stage = None,
    section = None,
    experiment = None,
    parameter = None,
    metric = None,
    value = None,
    status = None,
    decision = None,
    notes = None):

    global insights_repository

    new_record = pd.DataFrame([{
        "stage": stage,
        "section": section,
        "experiment": experiment,
        "parameter": parameter,
        "metric": metric,
        "value": value,
        "status": status,
        "decision": decision,
        "insight": insight,
        "notes": notes
    }])

    insights_repository = pd.concat(
        [insights_repository, new_record],
        ignore_index = True
    )

    print_outputs("New Insight Added to Repository")

    if stage:
        print(f"Stage      : {stage}")
    if section:
        print(f"Section    : {section}")
    if experiment:
        print(f"Experiment : {experiment}")
    if parameter:
        print(f"Parameter  : {parameter}")
    if metric:
        print(f"Metric     : {metric}")
    if value is not None:
        print(f"Value      : {value}")
    if status:
        print(f"Status     : {status}")
    if decision:
        print(f"Decision   : {textwrap.fill(str(decision), width = 75)}")

    print(f"\nInsight:  {textwrap.fill(str(insight), width = 75)}")

    if notes:
        print(f"\nNotes: {textwrap.fill(str(notes), width = 75)}")

    print("--" * 40)


# Create function to store information in function repository
function_repository = pd.DataFrame(
    columns = ["Function", "Description", "Input", "Output"]
)


def store_function(function, description = None, input = None, output = None):

    global function_repository

    function_name = function.__name__

    new_row = {
        "Function": function_name,
        "Description": description,
        "Input": str(inspect.signature(function)),
        "Output": output
    }

    if function_name in function_repository["Function"].values:

        for column, value in new_row.items():
            function_repository.loc[
                function_repository["Function"] == function_name,
                column
            ] = value

        print_outputs(
            f"Function {function_name} has been updated in the function_repository."
        )

    else:

        function_repository = pd.concat(
            [function_repository, pd.DataFrame([new_row])],
            ignore_index = True
        )

        print_outputs(
            f"Function '{function_name}' has been added to the function_repository."
        )


# Store repository functions
store_function(
    print_outputs,
    description = "Returns notebook outputs in a consistent organized format."
)

store_function(
    store_insight,
    description = "Stores decisions, findings, and interpretations for later reporting."
)

store_function(
    store_function,
    description = "Stores reusable functions and their signatures for traceability."
)

display(function_repository)

Function 'print_outputs' has been added to the function_repository.
Function 'store_insight' has been added to the function_repository.
Function 'store_function' has been added to the function_repository.


,Function,Description,Input,Output
0,print_outputs,Returns notebook outputs in a consistent organized format.,"(title=None, *outputs)",None
1,store_insight,"Stores decisions, findings, and interpretations for later reporting.","(insight, stage=None, section=None, experiment=None, parameter=None, metric=None, value=None, status=None, decision=...",None
2,store_function,Stores reusable functions and their signatures for traceability.,"(function, description=None, input=None, output=None)",None


## Task 2: Loading Data
----------------

In [3]:
# Provide folder path for data
statistics_folder = Path("Data/Statistical Analysis")

evidence_repository = pd.read_csv(
    statistics_folder / "evidence_repository.csv",
    dtype = {"Facility ID": str})

# Check the combined evidence repository
print(line)
print("Shape of Evidence Repository: ", evidence_repository.shape)
print("Number of Unique Facilities: ", evidence_repository["Facility ID"].nunique())
print("Evidence Repository Columns: ", evidence_repository.columns.tolist())
print(line)

display(evidence_repository.head())

--------------------------------------------------------------------------------
Shape of Evidence Repository:  (57298, 14)
Number of Unique Facilities:  1646
Evidence Repository Columns:  ['Facility ID', 'Facility Name', 'Domain', 'Condition', 'Measure ID', 'Measure Name', 'Score', 'Units', 'Direction', 'Benchmark Cohort Mean', 'Cohort Std', 'Difference From Cohort Mean', 'Relative Position to Cohort', 'Compared to National']
--------------------------------------------------------------------------------


,Facility ID,Facility Name,Domain,Condition,Measure ID,Measure Name,Score,Units,Direction,Benchmark Cohort Mean,Cohort Std,Difference From Cohort Mean,Relative Position to Cohort,Compared to National
0,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_1_STAR_RATING,Nurse communication,3.0,Stars,Higher is Better,3.03,0.910164,-0.03,Similar to Cohort Mean,NaN
1,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_2_STAR_RATING,Doctor communication,4.0,Stars,Higher is Better,2.97,0.676761,1.03,Above Cohort Mean,NaN
2,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_5_STAR_RATING,Communication about medicines,2.0,Stars,Higher is Better,2.25,0.684553,-0.25,Similar to Cohort Mean,NaN
3,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_6_STAR_RATING,Discharge information,3.0,Stars,Higher is Better,3.10,0.897424,-0.10,Similar to Cohort Mean,NaN
4,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_CLEAN_STAR_RATING,Cleanliness,3.0,Stars,Higher is Better,3.00,0.878655,-0.00,Similar to Cohort Mean,NaN


## Task 3: Assess Evidence Repository
------------------

In [4]:
# Validate evidence repository structure and completeness

def validate_evidence_repository(evidence_repository):

    evidence = evidence_repository.copy()

    core_columns = [
        "Facility ID", "Facility Name", "Domain", "Measure ID", "Measure Name", "Score",
        "Units", "Direction", "Benchmark Cohort Mean", "Difference From Cohort Mean",
        "Relative Position to Cohort"
    ]

    print(line)
    print("Evidence Repository Validation")
    print(line)

    print("Shape:", evidence.shape)
    print("Unique Facilities:", evidence["Facility ID"].nunique())
    print("Unique Measures:", evidence["Measure ID"].nunique())

    print("\nMissing Core Values:")
    display(evidence[core_columns].isna().sum().to_frame("Missing Values"))

    print("\nDataset-Specific Missing Values:")
    display(evidence[["Condition", "Compared to National"]].isna().sum().to_frame("Expected Missing Values"))
    print(line)

    duplicate_count = evidence.duplicated(subset = ["Facility ID", "Domain", "Measure ID"]).sum()

    print("Duplicate Facility-Domain-Measure Rows:", duplicate_count)

    numeric_score = pd.to_numeric(evidence["Score"], errors = "coerce")
    print("Non-Numeric Scores:", numeric_score.isna().sum())
    print(line)

    print("\nMeasures by Domain:")
    display(evidence.groupby("Domain")["Measure ID"].nunique().to_frame("Unique Measures"))
    print(line)

    print("\nFacilities by Domain:")
    display(evidence.groupby("Domain")["Facility ID"].nunique().to_frame("Unique Facilities"))

    print(line)


store_function(
    function = validate_evidence_repository,
    description = "Validates core evidence fields while separating expected dataset-specific missing values."
)

Function 'validate_evidence_repository' has been added to the function_repository.


In [5]:
# Implementation: validate_evidence_repository

evidence_validation = validate_evidence_repository(evidence_repository)


--------------------------------------------------------------------------------
Evidence Repository Validation
--------------------------------------------------------------------------------
Shape: (57298, 14)
Unique Facilities: 1646
Unique Measures: 37

Missing Core Values:


,Missing Values
Facility ID,0
Facility Name,0
Domain,0
Measure ID,0
Measure Name,0
Score,0
Units,0
Direction,0
Benchmark Cohort Mean,0
Difference From Cohort Mean,0



Dataset-Specific Missing Values:


,Expected Missing Values
Condition,42792
Compared to National,29320


--------------------------------------------------------------------------------
Duplicate Facility-Domain-Measure Rows: 0
Non-Numeric Scores: 0
--------------------------------------------------------------------------------

Measures by Domain:


,Unique Measures
Domain,
Healthcare-Associated Infections,18
Patient Survey,9
Timely and Effective Care,10


--------------------------------------------------------------------------------

Facilities by Domain:


,Unique Facilities
Domain,
Healthcare-Associated Infections,1646
Patient Survey,1646
Timely and Effective Care,1646


--------------------------------------------------------------------------------


In [6]:
# Store insight
store_insight(
    insight = """
Evidence repository validation passed and is ready for narrative generation.
Core statistical fields are complete, numeric, and consistently structured across
facilities and measures. Missing values in Compared to National are acceptable
because this field is only available for applicable CMS measures and serves as
supplemental national context rather than the primary cohort comparison.
"""
)

New Insight Added to Repository

Insight:   Evidence repository validation passed and is ready for narrative
generation. Core statistical fields are complete, numeric, and consistently
structured across facilities and measures. Missing values in Compared to
National are acceptable because this field is only available for applicable
CMS measures and serves as supplemental national context rather than the
primary cohort comparison.
--------------------------------------------------------------------------------


In [7]:
# Store evidence repository validation insight

store_insight(
    insight = """
Condition and Compared to National are dataset-specific fields. Condition applies
to Timely and Effective Care, while Compared to National applies to HAI measures;
therefore, missing values outside their respective domains are expected.
"""
)

New Insight Added to Repository

Insight:   Condition and Compared to National are dataset-specific fields. Condition
applies to Timely and Effective Care, while Compared to National applies to
HAI measures; therefore, missing values outside their respective domains
are expected.
--------------------------------------------------------------------------------


## Task 4: Providing Contextual Information
----------------------------

#### Parameters

In [8]:
# Create dictionary to store narrative, retrieval, and evaluation parameters

parameters = {

    "narrative": {
        "token_limit": 512,
        "max_measures": 8,
        "max_evidence_per_domain": 8
    },

    "retrieval": {
        "facility_columns": ["Facility ID", "Facility Name"],
        "filter_columns": ["Domain", "Measure ID", "Measure Name", "Direction"]
    },

    "evaluation": {
        "sample_size": 5,
        "random_seed": 42,
        "splits": {
            "training": 0.60,
            "development": 0.20,
            "testing": 0.20
        }
    }
}


In [9]:
# Create concise dataset-level context for narrative generation

dataset_context = {

    "Patient Survey": {"description": "Measures patient-reported experiences during inpatient hospital stays."},

    "Healthcare-Associated Infections": {"description": "Measures healthcare-associated infections and related exposure or volume information."},

    "Timely and Effective Care": {"description": "Measures how quickly or effectively hospitals provide recommended care."}
}

In [10]:
# Create dictionary to define evidence relevance by audience

audience_relevance = {

    "Executive Summary": {
        "Patient Survey": ["All"],
        "Healthcare-Associated Infections": ["SIR", "Observed Cases", "Predicted Cases", "Procedures"],
        "Timely and Effective Care": ["All"]
    },

    "Clinical": {
        "Patient Survey": ["All"],
        "Healthcare-Associated Infections": ["All"],
        "Timely and Effective Care": ["All"]
    },

    "Patient Friendly": {
        "Patient Survey": ["All"],
        "Healthcare-Associated Infections": ["SIR"],
        "Timely and Effective Care": ["All"]
    },

    "Community Report": {
        "Patient Survey": ["All"],
        "Healthcare-Associated Infections": ["SIR"],
        "Timely and Effective Care": ["All"]
    }
}

print_outputs(
    "Audience Evidence Relevance",
    ("Audience Relevance", audience_relevance)
)

Audience Evidence Relevance
Audience Relevance
{'Executive Summary': {'Patient Survey': ['All'], 'Healthcare-Associated Infections': ['SIR', 'Observed Cases', 'Predicted Cases', 'Procedures'], 'Timely and Effective Care': ['All']}, 'Clinical': {'Patient Survey': ['All'], 'Healthcare-Associated Infections': ['All'], 'Timely and Effective Care': ['All']}, 'Patient Friendly': {'Patient Survey': ['All'], 'Healthcare-Associated Infections': ['SIR'], 'Timely and Effective Care': ['All']}, 'Community Report': {'Patient Survey': ['All'], 'Healthcare-Associated Infections': ['SIR'], 'Timely and Effective Care': ['All']}}
--------------------------------------------------------------------------------


In [11]:
# Version 1 is more direct, Version 2 uses audience/scenario framing, and Version 3 gives explicit output structure

# Create dictionary with various communication style prompts:

communication_prompts = {

    "Patient Friendly": [
        (
            "Rewrite the hospital information for a patient with no healthcare background. "
            "Use plain everyday language, short sentences, and explain unfamiliar terms simply. "
            "Preserve all numbers, comparisons, and performance meaning."
        ),
        (
            "Imagine you are explaining these hospital results to a patient deciding where to receive care. "
            "Focus on what the results could mean to them, avoid jargon, and make strengths and weaknesses easy to follow. "
            "Do not add facts or change any statistics."
        ),
        (
            "Create a simple patient-facing summary with three parts: overall strength, key positive results, and areas that may need improvement. "
            "Use clear nontechnical wording and preserve all numerical evidence and performance interpretations."
        )
    ],

    "Executive Summary": [
        (
            "Rewrite the hospital information as a concise executive briefing for hospital leadership. "
            "Prioritize the strongest domain, major risks, and the most important benchmark comparisons. "
            "Preserve all quantitative evidence."
        ),
        (
            "Present these results to a hospital executive who needs to quickly understand where performance is strongest and where attention may be needed. "
            "Use direct, strategic language and focus on relative performance rather than explaining every measure."
        ),
        (
            "Create a leadership summary organized around performance priorities: strongest area, supporting evidence, weakest area, and key improvement opportunities. "
            "Keep the tone professional and preserve all reported numerical findings."
        )
    ],

    "Clinical": [
        (
            "Rewrite the hospital information for healthcare professionals. "
            "Use appropriate clinical terminology, retain measure-specific detail, and report quantitative comparisons precisely. "
            "Do not infer causation or statistical significance."
        ),
        (
            "Present these findings as a clinical performance review for physicians and quality staff. "
            "Focus on measure results, cohort comparisons, direction of performance, and clinically relevant areas for follow-up. "
            "Do not add unsupported conclusions."
        ),
        (
            "Create a structured clinical narrative that identifies the strongest and weakest performance areas, then supports each with the most relevant measures and benchmark values. "
            "Preserve clinical terminology, scores, units, and comparison meaning."
        )
    ],

    "Community Report": [
        (
            "Rewrite the hospital information for a public community report. "
            "Use accessible, neutral language and explain the main strengths and weaknesses without technical detail. "
            "Preserve all important numerical facts."
        ),
        (
            "Imagine these results will appear in a community hospital report for local residents. "
            "Explain what the hospital appears to do well and where improvement may be needed using balanced, understandable language. "
            "Do not exaggerate or add new facts."
        ),
        (
            "Create a short community-facing report organized around overall strengths, notable results, and improvement areas. "
            "Use clear public-facing language, mention important comparisons, and preserve all factual values and performance meaning."
        )
    ]
}

### Functions

In [12]:
# Create function to select narrative evidence for a facility and audience

def select_audience_evidence(evidence_repository, facility_id, audience, audience_relevance, parameters):

    facility = evidence_repository[evidence_repository["Facility ID"] == facility_id].copy()

    if facility.empty:
        raise ValueError(f"No evidence found for Facility ID: {facility_id}")

    max_measures = parameters["narrative"]["max_measures"]
    selected = []

    for domain, group in facility.groupby("Domain"):
        relevance = audience_relevance.get(audience, {}).get(domain, [])

        if "All" in relevance:
            domain_evidence = group.copy()
        else:
            domain_evidence = group[group["Measure ID"].astype(str).apply(
                lambda value: any(term in value for term in relevance))].copy()

        selected.append(domain_evidence.head(max_measures))

    return pd.concat(selected, ignore_index=True)


store_function(
    function = select_audience_evidence,
    description = "Selects facility evidence relevant to a specified narrative audience."
)

Function 'select_audience_evidence' has been added to the function_repository.


In [13]:
# Function Testing: select_audience_evidence

sample_facility = evidence_repository["Facility ID"].iloc[0]

sample_evidence = select_audience_evidence(
    evidence_repository,
    facility_id = sample_facility,
    audience = "Executive Summary",
    audience_relevance = audience_relevance,
    parameters = parameters
)

display(sample_evidence)

,Facility ID,Facility Name,Domain,Condition,Measure ID,Measure Name,Score,Units,Direction,Benchmark Cohort Mean,Cohort Std,Difference From Cohort Mean,Relative Position to Cohort,Compared to National
0,010005,MARSHALL MEDICAL CENTERS,Healthcare-Associated Infections,NaN,HAI_1_SIR,CLABSI (ICU + select Wards),1.69,Ratio (Observed/Predicted Cases),Lower is Better,0.55,0.511284,1.14,Above Cohort Mean,No Different than National Benchmark
1,010005,MARSHALL MEDICAL CENTERS,Healthcare-Associated Infections,NaN,HAI_2_SIR,CAUTI (ICU + select Wards),1.72,Ratio (Observed/Predicted Cases),Lower is Better,0.50,0.468985,1.22,Above Cohort Mean,No Different than National Benchmark
2,010005,MARSHALL MEDICAL CENTERS,Healthcare-Associated Infections,NaN,HAI_3_SIR,SSI (Colon Surgery),0.92,Ratio (Observed/Predicted Cases),Lower is Better,0.83,0.676686,0.09,Similar to Cohort Mean,No Different than National Benchmark
3,010005,MARSHALL MEDICAL CENTERS,Healthcare-Associated Infections,NaN,HAI_5_SIR,MRSA Bacteremia,1.54,Ratio (Observed/Predicted Cases),Lower is Better,0.69,0.570544,0.86,Above Cohort Mean,No Different than National Benchmark
4,010005,MARSHALL MEDICAL CENTERS,Healthcare-Associated Infections,NaN,HAI_6_SIR,Clostridium Difficile,0.42,Ratio (Observed/Predicted Cases),Lower is Better,0.34,0.242738,0.09,Similar to Cohort Mean,No Different than National Benchmark
5,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_1_STAR_RATING,Nurse communication,3.00,Stars,Higher is Better,3.03,0.910164,-0.03,Similar to Cohort Mean,NaN
6,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_2_STAR_RATING,Doctor communication,4.00,Stars,Higher is Better,2.97,0.676761,1.03,Above Cohort Mean,NaN
7,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_5_STAR_RATING,Communication about medicines,2.00,Stars,Higher is Better,2.25,0.684553,-0.25,Similar to Cohort Mean,NaN
8,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_6_STAR_RATING,Discharge information,3.00,Stars,Higher is Better,3.10,0.897424,-0.10,Similar to Cohort Mean,NaN
9,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_CLEAN_STAR_RATING,Cleanliness,3.00,Stars,Higher is Better,3.00,0.878655,-0.00,Similar to Cohort Mean,NaN


In [14]:
# Create function to separate performance and contextual measures

def separate_measure_roles(dataframe):

    performance_measures = dataframe[
        dataframe["Direction"].isin(["Higher is Better", "Lower is Better"])].copy()

    context_columns = ["Facility ID", "Facility Name", "Domain", "Condition", "Measure ID",
                       "Measure Name", "Score", "Units"]

    contextual_measures = dataframe[
        dataframe["Direction"] == "Context Only"][context_columns].copy()

    return performance_measures, contextual_measures

store_function(
    function = separate_measure_roles,
    description = "Separates performance measures for statistical ranking from simplified contextual measures for narrative support."
)

Function 'separate_measure_roles' has been added to the function_repository.


In [15]:
# Create function to rank performance measures and domains

def add_performance_rankings(dataframe):

    ranked = dataframe[dataframe["Direction"].isin(["Higher is Better", "Lower is Better"])].copy()

    ranked["Standardized Difference"] = (
        ranked["Difference From Cohort Mean"] / ranked["Cohort Std"]
    )

    lower_mask = ranked["Direction"] == "Lower is Better"
    ranked.loc[lower_mask, "Standardized Difference"] *= -1

    ranked["Measure Ranking"] = ranked.groupby("Facility ID")["Standardized Difference"].rank(
        method="min", ascending=False).astype("Int64")

    domain_score = ranked.groupby(["Facility ID", "Domain"])["Standardized Difference"].transform("mean")

    ranked["Domain Ranking"] = domain_score.groupby(ranked["Facility ID"]).rank(
        method="dense", ascending=False).astype("Int64")

    return ranked


store_function(
    function = add_performance_rankings,
    description = "Ranks individual measures and overall domains using direction-adjusted standardized performance."
)

Function 'add_performance_rankings' has been added to the function_repository.


In [16]:
# Create function to summarize facility-level performance evidence

def build_performance_summary(performance_rankings, top_n=2):

    summaries = []

    for facility_id, facility in performance_rankings.groupby("Facility ID"):
        facility_name = facility["Facility Name"].iloc[0]

        domain_summary = facility.groupby("Domain", as_index=False).agg(
            Average_Standardized_Performance=("Standardized Difference", "mean"))

        domain_summary["Domain Ranking"] = domain_summary[
            "Average_Standardized_Performance"].rank(method="dense", ascending=False).astype("Int64")

        strongest_domain = domain_summary.nsmallest(1, "Domain Ranking").iloc[0]
        weakest_domain = domain_summary.nlargest(1, "Domain Ranking").iloc[0]

        strongest_rows = facility[facility["Domain"] == strongest_domain["Domain"]]
        weakest_rows = facility[facility["Domain"] == weakest_domain["Domain"]]

        strongest_measures = strongest_rows.nlargest(top_n, "Standardized Difference")
        weakest_measures = weakest_rows.nsmallest(top_n, "Standardized Difference")

        national = strongest_rows["Compared to National"].dropna()
        national_comparison = national.mode().iloc[0] if not national.mode().empty else None

        measure_columns = ["Measure Name", "Score", "Units", "Benchmark Cohort Mean",
                           "Difference From Cohort Mean", "Direction"]

        summaries.append({
            "Facility ID": facility_id,
            "Facility Name": facility_name,
            "Strongest Domain": strongest_domain["Domain"],
            "Strongest Domain Performance": round(strongest_domain["Average_Standardized_Performance"], 2),
            "Strongest Domain National Comparison": national_comparison,
            "Strongest Measures": strongest_measures[measure_columns].to_dict("records"),
            "Weakest Domain": weakest_domain["Domain"],
            "Weakest Domain Performance": round(weakest_domain["Average_Standardized_Performance"], 2),
            "Weakest Measures": weakest_measures[measure_columns].to_dict("records")
        })

    return pd.DataFrame(summaries)



store_function(
    function = build_performance_summary,
    description = "Summarizes strongest and weakest facility domains with benchmark context for key measures."
)

Function 'build_performance_summary' has been added to the function_repository.


In [17]:
# Create function to build compact contextual evidence

def build_context_evidence(contextual_measures, facility_id, max_context=2):

    context = contextual_measures[contextual_measures["Facility ID"] == facility_id].copy()

    if context.empty:
        return ""

    context = context.head(max_context)
    lines = ["Context:"]

    for domain, group in context.groupby("Domain", sort=False):
        lines.append(f"  {domain}")

        for _, row in group.iterrows():
            lines.append(f"    - {row['Measure Name']}: {row['Score']} {row['Units']}")

    return "\n".join(lines)



store_function(
    function = build_context_evidence,
    description = "Formats a small number of contextual measures for supplemental narrative evidence."
)

Function 'build_context_evidence' has been added to the function_repository.


In [18]:
# Create function to build narrative-ready facility evidence
def build_narrative_evidence(summary_row):

    evidence = f"Facility: {summary_row['Facility Name']}\n\nStrengths:\n"
    evidence += f"  {summary_row['Strongest Domain']}\n"

    if pd.notna(summary_row["Strongest Domain National Comparison"]):
        evidence += f"  Overall: {summary_row['Strongest Domain National Comparison']}\n"

    evidence += f"  Relative Performance: {summary_row['Strongest Domain Performance']:.2f} SD from cohort\n"
    evidence += "\n  Key Measures:\n"

    for measure in summary_row["Strongest Measures"]:
        evidence += f"    - {measure['Measure Name']}: {measure['Score']:.2f} {measure['Units']}\n"
        evidence += f"      Cohort Mean: {measure['Benchmark Cohort Mean']:.2f} | {measure['Direction']}\n"

    evidence += f"\nWeaknesses:\n  {summary_row['Weakest Domain']}\n"
    evidence += "\n  Key Measures:\n"

    for measure in summary_row["Weakest Measures"]:
        evidence += f"    - {measure['Measure Name']}: {measure['Score']:.2f} {measure['Units']}\n"
        evidence += f"      Cohort Mean: {measure['Benchmark Cohort Mean']:.2f} | {measure['Direction']}\n"

    return evidence.strip()


store_function(
    function = build_narrative_evidence,
    description = "Builds concise facility evidence with benchmark context for grounded narrative generation."
)

Function 'build_narrative_evidence' has been added to the function_repository.


In [19]:
# Create function to build narrative generation prompt

def build_narrative_prompt(evidence, context, style, communication_prompts, version=1):

    instructions = communication_prompts[style][version - 1]

    instructions += (
        " Use contextual measures only as supporting information, not as indicators of performance. "
        "Use only the evidence provided."
    )

    prompt = f"{instructions}\n\nPerformance Evidence:\n{evidence}"

    if context:
        prompt += f"\n\nSupporting Context:\n{context}"

    prompt += "\n\nNarrative:"

    return prompt


store_function(
    function = build_narrative_prompt,
    description = "Creates versioned FLAN-T5 prompts using communication style, performance evidence, and contextual support."
)

Function 'build_narrative_prompt' has been added to the function_repository.


In [20]:
# Create function to prepare facility evidence and prompt

def prepare_facility_prompt(evidence_repository, facility_id, audience, audience_relevance,
                            parameters, communication_prompts, version=1):

    selected = select_audience_evidence(
        evidence_repository=evidence_repository,
        facility_id=facility_id,
        audience=audience,
        audience_relevance=audience_relevance,
        parameters=parameters
    )

    performance_evidence, contextual_evidence = separate_measure_roles(selected)

    if performance_evidence.empty:
        raise ValueError(f"No performance evidence found for Facility ID {facility_id}.")

    performance_rankings = add_performance_rankings(performance_evidence)
    performance_summary = build_performance_summary(performance_rankings, top_n=2)

    if performance_summary.empty:
        raise ValueError(f"No performance summary created for Facility ID {facility_id}.")

    summary_row = performance_summary.iloc[0]
    context = build_context_evidence(contextual_evidence, facility_id=facility_id, max_context=2)
    evidence = build_narrative_evidence(summary_row)

    prompt = build_narrative_prompt(
        evidence=evidence,
        context=context,
        style=audience,
        communication_prompts=communication_prompts,
        version=version
    )

    return {
        "Facility ID": facility_id,
        "Facility Name": summary_row["Facility Name"],
        "Audience": audience,
        "Prompt Version": version,
        "Performance Evidence": evidence,
        "Context Evidence": context,
        "Prompt": prompt
    }

store_function(function = prepare_facility_prompt)

Function 'prepare_facility_prompt' has been added to the function_repository.


In [21]:
# Testing: prepare facility prompts from a random sample

# Select 5 random facilities
facility_sample = evidence_repository["Facility ID"].drop_duplicates().sample(
    n = 5, random_state = 42)

# Filter evidence to sampled facilities
evidence_sample = evidence_repository[
    evidence_repository["Facility ID"].isin(facility_sample)
].copy()

# Prepare prompts
prompt_samples = []
failed_facilities = []

for facility_id in facility_sample:

    result = prepare_facility_prompt(
        evidence_repository = evidence_sample,
        facility_id = facility_id,
        audience = "Executive Summary",
        audience_relevance = audience_relevance,
        parameters = parameters,
        communication_prompts = communication_prompts,
        version = 1)

    if result is not None:
        prompt_samples.append(result)
    else:
        failed_facilities.append(facility_id)

# Display testing summary
print_outputs(
    "Facility Prompt Test",
    ("Successful Prompts", len(prompt_samples)),
    ("Failed Facilities", failed_facilities)
)

# Display first successful prompt
if prompt_samples:

    sample_result = prompt_samples[0]

    print_outputs(
        "Sample Facility Prompt",
        ("Facility", sample_result["Facility Name"]),
        ("Audience", sample_result["Audience"]),
        ("Prompt Version", sample_result["Prompt Version"]),
        ("Performance Evidence", sample_result["Performance Evidence"]),
        ("Context Evidence", sample_result["Context Evidence"]),
        ("Final Prompt", sample_result["Prompt"])
    )

Facility Prompt Test
Successful Prompts
5
--------------------------------------------------------------------------------
Failed Facilities
[]
--------------------------------------------------------------------------------
Sample Facility Prompt
Facility
NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER
--------------------------------------------------------------------------------
Audience
Executive Summary
--------------------------------------------------------------------------------
Prompt Version
1
--------------------------------------------------------------------------------
Performance Evidence
Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER

Strengths:
  Timely and Effective Care
  Relative Performance: 0.06 SD from cohort

  Key Measures:
    - ED Time - Psychiatric Patients: 106.00 Minutes
      Cohort Mean: 319.04 | Lower is Better
    - Septic Shock 3-Hour Bundle: 87.00 Percent
      Cohort Mean: 71.21 | Higher is Better

Weaknesses:
  Patient Survey

  Key Measures:


In [22]:
# Initialize tokenizer

tokenizer_model = "google/flan-t5-large"

# Initialize tokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)

In [23]:
# Create function to count evidence tokens

def count_tokens(text, tokenizer):

    return len(tokenizer.encode(text))


store_function(
    function = count_tokens,
    description = "Counts the number of tokens in narrative evidence or prompts."
)

Function 'count_tokens' has been added to the function_repository.


In [24]:
# Testing: check compact evidence token count

evidence_tokens = count_tokens(sample_result['Prompt'], tokenizer)

print("Evidence Tokens:", evidence_tokens)
print("Evidence Token Limit:", parameters["narrative"]["token_limit"])
print("Within Limit:", evidence_tokens <= parameters["narrative"]["token_limit"])

Evidence Tokens: 224
Evidence Token Limit: 512
Within Limit: True


### Implementation

In [25]:
# Implementation: Create facility sample directly from evidence repository

facility_sample = evidence_repository[["Facility ID", "Facility Name"]].drop_duplicates()
facility_sample = facility_sample.sample(n=min(10, len(facility_sample)),
                                         random_state=parameters["evaluation"]["random_seed"])
facility_sample = facility_sample.reset_index(drop=True)

print_outputs("Facility Sample", ("Source", "Evidence Repository"), ("Facilities", len(facility_sample)))

display(facility_sample)

Facility Sample
Source
Evidence Repository
--------------------------------------------------------------------------------
Facilities
10
--------------------------------------------------------------------------------


,Facility ID,Facility Name
0,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER
1,500072,OLYMPIC MEDICAL CENTER
2,220046,BERKSHIRE MEDICAL CENTER
3,360076,ATRIUM MEDICAL CENTER
4,390145,EXCELA HEALTH WESTMORELAND REGIONAL HOSPITAL
5,050498,SUTTER AUBURN FAITH HOSPITAL
6,490107,RESTON HOSPITAL CENTER
7,240075,ESSENTIA HEALTH ST JOSEPH'S MEDICAL CENTER
8,390046,WELLSPAN YORK HOSPITAL
9,290041,SUMMERLIN HOSPITAL MEDICAL CENTER


In [26]:
# Implementation: Prepare prompts for sampled facilities

prepared_prompt_rows = []
failed_prompt_rows = []

for _, facility in facility_sample.iterrows():
    facility_id = facility["Facility ID"]

    for audience in communication_prompts.keys():
        try:
            result = prepare_facility_prompt(evidence_repository=evidence_repository,
                                             facility_id=facility_id,
                                             audience=audience,
                                             audience_relevance=audience_relevance,
                                             parameters=parameters,
                                             communication_prompts=communication_prompts,
                                             version=1)

            prepared_prompt_rows.append({
                "Facility ID": result["Facility ID"],
                "Facility Name": result["Facility Name"],
                "Audience": result["Audience"],
                "Prompt Version": result["Prompt Version"],
                "Performance Evidence": result["Performance Evidence"],
                "Context Evidence": result["Context Evidence"],
                "Prompt": result["Prompt"]
            })

        except Exception as error:
            failed_prompt_rows.append({
                "Facility ID": facility_id,
                "Audience": audience,
                "Error": str(error)
            })

prepared_prompts = pd.DataFrame(prepared_prompt_rows)
failed_prompts = pd.DataFrame(failed_prompt_rows)

print_outputs("Prepared Facility Prompts",
              ("Facilities", prepared_prompts["Facility ID"].nunique()),
              ("Audiences", prepared_prompts["Audience"].nunique()),
              ("Prepared Prompts", len(prepared_prompts)),
              ("Failed Prompts", len(failed_prompts)))

display(prepared_prompts.head(8))

Prepared Facility Prompts
Facilities
10
--------------------------------------------------------------------------------
Audiences
4
--------------------------------------------------------------------------------
Prepared Prompts
40
--------------------------------------------------------------------------------
Failed Prompts
0
--------------------------------------------------------------------------------


,Facility ID,Facility Name,Audience,Prompt Version,Performance Evidence,Context Evidence,Prompt
0,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Patient Friendly,1,Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER\n\nStrengths:\n Timely and Effective Care\n Relative Perform...,,"Rewrite the hospital information for a patient with no healthcare background. Use plain everyday language, short sen..."
1,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Executive Summary,1,Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER\n\nStrengths:\n Timely and Effective Care\n Relative Perform...,,Rewrite the hospital information as a concise executive briefing for hospital leadership. Prioritize the strongest d...
2,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Clinical,1,Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER\n\nStrengths:\n Timely and Effective Care\n Relative Perform...,Context:\n Healthcare-Associated Infections\n - CLABSI (ICU + select Wards): 17212.0 Device Days\n - CLABSI (...,"Rewrite the hospital information for healthcare professionals. Use appropriate clinical terminology, retain measure-..."
3,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Community Report,1,Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER\n\nStrengths:\n Timely and Effective Care\n Relative Perform...,,"Rewrite the hospital information for a public community report. Use accessible, neutral language and explain the mai..."
4,500072,OLYMPIC MEDICAL CENTER,Patient Friendly,1,Facility: OLYMPIC MEDICAL CENTER\n\nStrengths:\n Patient Survey\n Relative Performance: 0.24 SD from cohort\n\n K...,,"Rewrite the hospital information for a patient with no healthcare background. Use plain everyday language, short sen..."
5,500072,OLYMPIC MEDICAL CENTER,Executive Summary,1,Facility: OLYMPIC MEDICAL CENTER\n\nStrengths:\n Patient Survey\n Relative Performance: 0.24 SD from cohort\n\n K...,,Rewrite the hospital information as a concise executive briefing for hospital leadership. Prioritize the strongest d...
6,500072,OLYMPIC MEDICAL CENTER,Clinical,1,Facility: OLYMPIC MEDICAL CENTER\n\nStrengths:\n Patient Survey\n Relative Performance: 0.24 SD from cohort\n\n K...,Context:\n Healthcare-Associated Infections\n - CLABSI (ICU + select Wards): 1599.0 Device Days\n - CLABSI (I...,"Rewrite the hospital information for healthcare professionals. Use appropriate clinical terminology, retain measure-..."
7,500072,OLYMPIC MEDICAL CENTER,Community Report,1,Facility: OLYMPIC MEDICAL CENTER\n\nStrengths:\n Patient Survey\n Relative Performance: 0.24 SD from cohort\n\n K...,,"Rewrite the hospital information for a public community report. Use accessible, neutral language and explain the mai..."


In [27]:
# Double Check: Validate prepared facility prompts

expected_facilities = len(facility_sample)
expected_audiences = len(communication_prompts)
expected_prompts = expected_facilities * expected_audiences

actual_facilities = prepared_prompts["Facility ID"].nunique()
actual_audiences = prepared_prompts["Audience"].nunique()
actual_prompts = len(prepared_prompts)
missing_prompts = prepared_prompts["Prompt"].isna().sum()

duplicate_prompts = prepared_prompts.duplicated(
    subset=["Facility ID", "Audience", "Prompt Version"]
).sum()

validation_passed = (actual_facilities == expected_facilities
                     and actual_audiences == expected_audiences
                     and actual_prompts == expected_prompts
                     and missing_prompts == 0
                     and duplicate_prompts == 0)

print("PREPARED PROMPT VALIDATION")
print("=" * 70)
print(f"Expected Facilities: {expected_facilities} | Actual Facilities: {actual_facilities}")
print(f"Expected Audiences: {expected_audiences} | Actual Audiences: {actual_audiences}")
print(f"Expected Prompts: {expected_prompts} | Actual Prompts: {actual_prompts}")
print(f"Missing Prompts: {missing_prompts} | Duplicate Prompts: {duplicate_prompts}")
print("\nPrompts by Audience:")
print(prepared_prompts["Audience"].value_counts())
print(f"\nValidation Passed: {validation_passed}")

PREPARED PROMPT VALIDATION
Expected Facilities: 10 | Actual Facilities: 10
Expected Audiences: 4 | Actual Audiences: 4
Expected Prompts: 40 | Actual Prompts: 40
Missing Prompts: 0 | Duplicate Prompts: 0

Prompts by Audience:
Audience
Patient Friendly     10
Executive Summary    10
Clinical             10
Community Report     10
Name: count, dtype: int64

Validation Passed: True


In [28]:
# Double Check: Inspect one completed facility prompt

sample_prompt = prepared_prompts.iloc[0]

print(f"Sample Facility: {sample_prompt['Facility Name']}")
print(f"Facility ID: {sample_prompt['Facility ID']}")
print(f"Audience: {sample_prompt['Audience']}")
print(f"Prompt Version: {sample_prompt['Prompt Version']}")
print("\n" + "=" * 100)
print(sample_prompt["Prompt"])

Sample Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER
Facility ID: 200033
Audience: Patient Friendly
Prompt Version: 1

Rewrite the hospital information for a patient with no healthcare background. Use plain everyday language, short sentences, and explain unfamiliar terms simply. Preserve all numbers, comparisons, and performance meaning. Use contextual measures only as supporting information, not as indicators of performance. Use only the evidence provided.

Performance Evidence:
Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER

Strengths:
  Timely and Effective Care
  Relative Performance: 0.06 SD from cohort

  Key Measures:
    - ED Time - Psychiatric Patients: 106.00 Minutes
      Cohort Mean: 319.04 | Lower is Better
    - Septic Shock 3-Hour Bundle: 87.00 Percent
      Cohort Mean: 71.21 | Higher is Better

Weaknesses:
  Patient Survey

  Key Measures:
    - Recommend hospital: 2.00 Stars
      Cohort Mean: 3.41 | Higher is Better
    - Nurse communication: 2.00 Star

## Task 5: Tokenize
--------------------------------------------------------------------------


In [31]:
from transformers import AutoTokenizer

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [32]:
# Implementation: Calculate token counts for prepared prompts

prepared_prompts["Prompt Tokens"] = prepared_prompts["Prompt"].apply(lambda text: count_tokens(text, tokenizer))

display(prepared_prompts[["Facility ID", "Facility Name", "Audience", "Prompt Version", "Prompt Tokens"]])

,Facility ID,Facility Name,Audience,Prompt Version,Prompt Tokens
0,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Patient Friendly,1,224
1,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Executive Summary,1,224
2,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Clinical,1,274
3,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Community Report,1,219
4,500072,OLYMPIC MEDICAL CENTER,Patient Friendly,1,211
5,500072,OLYMPIC MEDICAL CENTER,Executive Summary,1,211
6,500072,OLYMPIC MEDICAL CENTER,Clinical,1,262
7,500072,OLYMPIC MEDICAL CENTER,Community Report,1,206
8,220046,BERKSHIRE MEDICAL CENTER,Patient Friendly,1,246
9,220046,BERKSHIRE MEDICAL CENTER,Executive Summary,1,246


In [ ]:
# Double Check: Identify prompts approaching input limit

print(prepared_prompts["Prompt Tokens"].describe())

max_input_tokens = 512

prepared_prompts["Within Token Limit"] = prepared_prompts["Prompt Tokens"] <= max_input_tokens

print("Prompts Within Limit:", prepared_prompts["Within Token Limit"].sum())
print("Prompts Over Limit:", (~prepared_prompts["Within Token Limit"]).sum())

display(prepared_prompts.loc[
    ~prepared_prompts["Within Token Limit"],
    ["Facility ID", "Facility Name", "Audience", "Prompt Tokens"]
])

count     40.000000
mean     248.950000
std       30.445369
min      206.000000
25%      222.750000
50%      246.000000
75%      263.000000
max      320.000000
Name: Prompt Tokens, dtype: float64
Prompts Within Limit: 40
Prompts Over Limit: 0


,Facility ID,Facility Name,Audience,Prompt Tokens


## Task 6: Generate Summaries
--------------------------------------------------------------------------


In [ ]:
# Implementation: Load FLAN-T5-large directly

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5-large model and tokenizer loaded.


In [41]:
# Create function to generate narrative summaries

def generate_summary(prompt, model, tokenizer, max_new_tokens=200):

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams = 2,
        do_sample = False,
        early_stopping = True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [40]:
# Test: Generate all communication styles for one facility

test_facility_id = prepared_prompts.iloc[0]["Facility ID"]

test_styles = prepared_prompts[
    prepared_prompts["Facility ID"] == test_facility_id
].copy()

print(f"Facility: {test_styles['Facility Name'].iloc[0]}")
print("=" * 100)

for _, row in test_styles.iterrows():

    output = generate_summary(
        prompt=row["Prompt"],
        model=model,
        tokenizer=tokenizer
    )

    print(f"\nCOMMUNICATION STYLE: {row['Audience']}")
    print(f"PROMPT TOKENS: {row['Prompt Tokens']}")
    print("-" * 100)
    print(output)

Facility: NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER

COMMUNICATION STYLE: Patient Friendly
PROMPT TOKENS: 224
----------------------------------------------------------------------------------------------------
NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER is a hospital that provides timely and effective care.

COMMUNICATION STYLE: Executive Summary
PROMPT TOKENS: 224
----------------------------------------------------------------------------------------------------
The facility at NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER has a strong focus on timely and effective care. The facility has a weak focus on patient satisfaction.

COMMUNICATION STYLE: Clinical
PROMPT TOKENS: 274
----------------------------------------------------------------------------------------------------
Healthcare-Associated Infections - CLABSI (ICU + select Wards): 17212.0 Device Days - CLABSI (ICU + select Wards): 9.0 Cases

COMMUNICATION STYLE: Community Report
PROMPT TOKENS: 219
------------------------------

In [42]:
# Implementation: Generate narratives for prepared prompts

generation_results = prepared_prompts.copy()

generation_results["Generated Narrative"] = generation_results["Prompt"].apply(
    lambda prompt: generate_summary(prompt, model, tokenizer))

display(generation_results[["Facility ID", "Facility Name", "Audience", "Prompt Tokens", "Generated Narrative"]].head(8))

,Facility ID,Facility Name,Audience,Prompt Tokens,Generated Narrative
0,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Patient Friendly,224,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER is a hospital that provides quality care. It has a low ED wait time and ...
1,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Executive Summary,224,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER is a hospital with strengths in timely and effective care. The hospital ...
2,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Clinical,274,Healthcare-Associated Infections - CLABSI (ICU + select Wards): 17212.0 Device Days - CLABSI (ICU + select Wards): 9...
3,200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,Community Report,219,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER is a hospital providing medical care in the Eastern Maine area. It has a...
4,500072,OLYMPIC MEDICAL CENTER,Patient Friendly,211,The OLYMPIC MEDICAL CENTER is a hospital with a high patient satisfaction rating. The hospital has a low ED time and...
5,500072,OLYMPIC MEDICAL CENTER,Executive Summary,211,The OLYMPIC MEDICAL CENTER has a strong patient survey and a strong nurse communication. The hospital has a weak tim...
6,500072,OLYMPIC MEDICAL CENTER,Clinical,262,Healthcare-Associated Infections - CLABSI (ICU + select Wards): 1599.0 Device Days - CLABSI (ICU + select Wards): 1....
7,500072,OLYMPIC MEDICAL CENTER,Community Report,206,OLYMPIC MEDICAL CENTER is a community hospital with a high level of patient satisfaction. The hospital has a high le...


#### Function


## Task 7: Training, Development, and Testing Samples
--------------------------------------------------------------------------

Due to computational limitations, each experimental split uses a sample of 10 facilities.
Facilities rather than individual measures are sampled so that all communication styles can be
generated from the same factual hospital narrative. The training split supports prompt
development, the development split supports prompt selection, and the testing split remains
held out for final evaluation.


In [43]:
# Implementation: Create reproducible facility-level experimental pools

RANDOM_SEED = parameters["evaluation"]["random_seed"]
sample_size = parameters["evaluation"]["sample_size"]

required_domains = ["Patient Survey", "Healthcare-Associated Infections", "Timely and Effective Care"]

facility_domain_counts = evidence_repository[
    evidence_repository["Domain"].isin(required_domains)
].groupby(["Facility ID", "Facility Name"])["Domain"].nunique().reset_index(name="Domain Count")

facility_pool = facility_domain_counts[
    facility_domain_counts["Domain Count"] == len(required_domains)
][["Facility ID", "Facility Name"]].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

n_facilities = len(facility_pool)
train_end = int(n_facilities * 0.60)
dev_end = int(n_facilities * 0.80)

train_pool = facility_pool.iloc[:train_end].copy()
dev_pool = facility_pool.iloc[train_end:dev_end].copy()
test_pool = facility_pool.iloc[dev_end:].copy()

print(f"Eligible Facilities: {len(facility_pool)}")
print(f"Training Pool: {len(train_pool)}")
print(f"Development Pool: {len(dev_pool)}")
print(f"Testing Pool: {len(test_pool)}")

Eligible Facilities: 1646
Training Pool: 987
Development Pool: 329
Testing Pool: 330


In [44]:
# Implementation: Sample facilities from each experimental pool

def create_split_sample(pool, split_name, sample_size=10):
    n = min(sample_size, len(pool))
    sample = pool.sample(n=n, random_state=RANDOM_SEED).copy()
    sample["Split"] = split_name
    return sample.reset_index(drop=True)

training_sample = create_split_sample(train_pool, "Training", sample_size)
development_sample = create_split_sample(dev_pool, "Development", sample_size)
testing_sample = create_split_sample(test_pool, "Testing", sample_size)

experiment_samples = pd.concat(
    [training_sample, development_sample, testing_sample], ignore_index=True
)

display(experiment_samples)

,Facility ID,Facility Name,Split
0,430077,MONUMENT HEALTH RAPID CITY HOSPITAL,Training
1,350015,SANFORD MEDICAL CENTER BISMARCK,Training
2,050549,LOS ROBLES HOSPITAL & MEDICAL CENTER,Training
3,050132,SAN GABRIEL VALLEY MEDICAL CENTER,Training
4,250123,SINGING RIVER GULFPORT,Training
5,150042,GOOD SAMARITAN HOSPITAL,Development
6,430014,AVERA ST LUKES,Development
7,190274,OCHSNER MEDICAL CENTER-KENNER,Development
8,380050,SKY LAKES MEDICAL CENTER,Development
9,190026,RAPIDES REGIONAL MEDICAL CENTER,Development


In [45]:
# Double Check: Confirm experimental splits do not overlap

train_ids = set(training_sample["Facility ID"])
dev_ids = set(development_sample["Facility ID"])
test_ids = set(testing_sample["Facility ID"])

train_dev_overlap = train_ids & dev_ids
train_test_overlap = train_ids & test_ids
dev_test_overlap = dev_ids & test_ids


print(f"Training Facilities: {len(training_sample)}")
print(f"Development Facilities: {len(development_sample)}")
print(f"Testing Facilities: {len(testing_sample)}")
print(f"Training/Development Overlap: {len(train_dev_overlap)}")
print(f"Training/Testing Overlap: {len(train_test_overlap)}")
print(f"Development/Testing Overlap: {len(dev_test_overlap)}")

split_valid = not train_dev_overlap and not train_test_overlap and not dev_test_overlap
print(f"\nSplit Validation Passed: {split_valid}")

Training Facilities: 5
Development Facilities: 5
Testing Facilities: 5
Training/Development Overlap: 0
Training/Testing Overlap: 0
Development/Testing Overlap: 0

Split Validation Passed: True


In [46]:
# Create function to prepare prompts for an experimental split

def prepare_split_prompts(split_sample, evidence_repository, communication_prompts):

    prepared_rows = []

    for _, facility in split_sample.iterrows():
        for audience in communication_prompts.keys():
            result = prepare_facility_prompt(
                evidence_repository=evidence_repository,
                facility_id=facility["Facility ID"],
                audience=audience,
                audience_relevance=audience_relevance,
                parameters=parameters,
                communication_prompts=communication_prompts,
                version=1
            )

            result["Split"] = facility["Split"]
            prepared_rows.append(result)

    return pd.DataFrame(prepared_rows)

In [47]:
# Implementation: Prepare prompts for training, development, and testing

training_prompts = prepare_split_prompts(
    training_sample, evidence_repository, communication_prompts
)

development_prompts = prepare_split_prompts(
    development_sample, evidence_repository, communication_prompts
)

testing_prompts = prepare_split_prompts(
    testing_sample, evidence_repository, communication_prompts
)

experiment_prompts = pd.concat(
    [training_prompts, development_prompts, testing_prompts], ignore_index=True
)

print(f"Training Prompts: {len(training_prompts)}")
print(f"Development Prompts: {len(development_prompts)}")
print(f"Testing Prompts: {len(testing_prompts)}")
print(f"Total Prompts: {len(experiment_prompts)}")

Training Prompts: 20
Development Prompts: 20
Testing Prompts: 20
Total Prompts: 60


In [48]:
# Store insight

store_insight(
    stage = "Narratives",
    section = "Experimental Sampling",
    parameter = "Sample Size",
    value = sample_size,
    status = "Selected",
    decision = "Use 10 facilities in each of the training, development, and testing samples.",
    insight = (
        "Computational limitations require a small experimental sample. Mutually exclusive "
        "facility pools are created before sampling so the same hospital cannot appear across "
        "training, development, and testing. Training supports prompt development, development "
        "supports prompt selection, and testing is reserved for final evaluation."
    )
)


New Insight Added to Repository
Stage      : Narratives
Section    : Experimental Sampling
Parameter  : Sample Size
Value      : 5
Status     : Selected
Decision   : Use 10 facilities in each of the training, development, and testing
samples.

Insight:  Computational limitations require a small experimental sample. Mutually
exclusive facility pools are created before sampling so the same hospital
cannot appear across training, development, and testing. Training supports
prompt development, development supports prompt selection, and testing is
reserved for final evaluation.
--------------------------------------------------------------------------------


## Task 10: Prepare Prompt Experiment
--------------------------------------------------------------------------

Each factual narrative is paired with every communication style and prompt variation. This
creates a compact experimental table that Notebook 5 can send to the selected language model.
Results should be summarized with plain-language tables showing the best prompt by style,
numeric preservation, readability, and factual consistency rather than large raw metric outputs.


In [49]:
# Implementation: Prepare prompt experiment across all experimental splits

def prepare_prompt_experiment(split_samples, evidence_repository, communication_prompts):

    experiment_rows = []
    failed_rows = []

    for split_name, split_sample in split_samples.items():

        for _, facility in split_sample.iterrows():
            facility_id = facility["Facility ID"]

            for audience in communication_prompts.keys():

                try:
                    result = prepare_facility_prompt(
                        evidence_repository=evidence_repository,
                        facility_id=facility_id,
                        audience=audience,
                        audience_relevance=audience_relevance,
                        parameters=parameters,
                        communication_prompts=communication_prompts,
                        version=1
                    )

                    result["Split"] = split_name
                    experiment_rows.append(result)

                except Exception as error:
                    failed_rows.append({
                        "Split": split_name,
                        "Facility ID": facility_id,
                        "Audience": audience,
                        "Error": str(error)
                    })

    prompt_experiment = pd.DataFrame(experiment_rows)
    failed_prompt_experiment = pd.DataFrame(failed_rows)

    return prompt_experiment, failed_prompt_experiment


split_samples = {
    "Training": training_sample,
    "Development": development_sample,
    "Testing": testing_sample
}

prompt_experiment, failed_prompt_experiment = prepare_prompt_experiment(
    split_samples=split_samples,
    evidence_repository=evidence_repository,
    communication_prompts=communication_prompts
)

print("PROMPT EXPERIMENT")
print("=" * 70)
print(f"Prepared Prompts: {len(prompt_experiment)}")
print(f"Failed Prompts: {len(failed_prompt_experiment)}")

display(prompt_experiment.head(8))

PROMPT EXPERIMENT
Prepared Prompts: 60
Failed Prompts: 0


,Facility ID,Facility Name,Audience,Prompt Version,Performance Evidence,Context Evidence,Prompt,Split
0,430077,MONUMENT HEALTH RAPID CITY HOSPITAL,Patient Friendly,1,Facility: MONUMENT HEALTH RAPID CITY HOSPITAL\n\nStrengths:\n Timely and Effective Care\n Relative Performance: 0....,,"Rewrite the hospital information for a patient with no healthcare background. Use plain everyday language, short sen...",Training
1,430077,MONUMENT HEALTH RAPID CITY HOSPITAL,Executive Summary,1,Facility: MONUMENT HEALTH RAPID CITY HOSPITAL\n\nStrengths:\n Timely and Effective Care\n Relative Performance: 0....,,Rewrite the hospital information as a concise executive briefing for hospital leadership. Prioritize the strongest d...,Training
2,430077,MONUMENT HEALTH RAPID CITY HOSPITAL,Clinical,1,Facility: MONUMENT HEALTH RAPID CITY HOSPITAL\n\nStrengths:\n Timely and Effective Care\n Relative Performance: 0....,Context:\n Healthcare-Associated Infections\n - CLABSI (ICU + select Wards): 6491.0 Device Days\n - CLABSI (I...,"Rewrite the hospital information for healthcare professionals. Use appropriate clinical terminology, retain measure-...",Training
3,430077,MONUMENT HEALTH RAPID CITY HOSPITAL,Community Report,1,Facility: MONUMENT HEALTH RAPID CITY HOSPITAL\n\nStrengths:\n Timely and Effective Care\n Relative Performance: 0....,,"Rewrite the hospital information for a public community report. Use accessible, neutral language and explain the mai...",Training
4,350015,SANFORD MEDICAL CENTER BISMARCK,Patient Friendly,1,Facility: SANFORD MEDICAL CENTER BISMARCK\n\nStrengths:\n Healthcare-Associated Infections\n Overall: No Different...,,"Rewrite the hospital information for a patient with no healthcare background. Use plain everyday language, short sen...",Training
5,350015,SANFORD MEDICAL CENTER BISMARCK,Executive Summary,1,Facility: SANFORD MEDICAL CENTER BISMARCK\n\nStrengths:\n Healthcare-Associated Infections\n Overall: No Different...,,Rewrite the hospital information as a concise executive briefing for hospital leadership. Prioritize the strongest d...,Training
6,350015,SANFORD MEDICAL CENTER BISMARCK,Clinical,1,Facility: SANFORD MEDICAL CENTER BISMARCK\n\nStrengths:\n Healthcare-Associated Infections\n Overall: Better than ...,Context:\n Healthcare-Associated Infections\n - CLABSI (ICU + select Wards): 7553.0 Device Days\n - CLABSI (I...,"Rewrite the hospital information for healthcare professionals. Use appropriate clinical terminology, retain measure-...",Training
7,350015,SANFORD MEDICAL CENTER BISMARCK,Community Report,1,Facility: SANFORD MEDICAL CENTER BISMARCK\n\nStrengths:\n Healthcare-Associated Infections\n Overall: No Different...,,"Rewrite the hospital information for a public community report. Use accessible, neutral language and explain the mai...",Training


In [50]:
# Double Check: Validate prompt experiment

prompt_counts = prompt_experiment.groupby(["Split", "Audience"]).size().unstack(fill_value=0)

print("PROMPTS BY SPLIT AND COMMUNICATION STYLE")
print("=" * 70)
display(prompt_counts)

print(f"Unique Facilities: {prompt_experiment['Facility ID'].nunique()}")
print(f"Missing Prompts: {prompt_experiment['Prompt'].isna().sum()}")
print(f"Duplicate Prompts: {prompt_experiment.duplicated(
    subset=['Split', 'Facility ID', 'Audience', 'Prompt Version']
).sum()}")

PROMPTS BY SPLIT AND COMMUNICATION STYLE


Audience,Clinical,Community Report,Executive Summary,Patient Friendly
Split,,,,
Development,5,5,5,5
Testing,5,5,5,5
Training,5,5,5,5


Unique Facilities: 15
Missing Prompts: 0
Duplicate Prompts: 0


## Task 11: Save Outputs
--------------------------------------------------------------------------


In [52]:
# Save narrative and experimental outputs

output_dir = Path("Data/Narratives")
output_dir.mkdir(parents=True, exist_ok=True)

evidence_repository.to_csv(output_dir / "evidence_repository.csv", index=False)

experiment_samples.to_csv(output_dir / "experiment_samples.csv", index=False)

prompt_experiment.to_csv(output_dir / "prompt_experiment.csv", index=False)

if "generation_results" in globals():
    generation_results.to_csv(output_dir / "generation_results.csv", index=False)

if "insights_repository" in globals():
    insights_repository.to_csv(output_dir / "narrative_insights.csv", index=False)

if "function_repository" in globals():
    function_repository.to_csv(output_dir / "function_repository.csv", index=False)

with open(output_dir / "communication_prompts.json", "w", encoding="utf-8") as file:
    json.dump(communication_prompts, file, indent=2)

print_outputs(
    "Notebook Complete",
    ("Output Folder", str(output_dir)),
    ("Training Sample", len(training_sample)),
    ("Development Sample", len(development_sample)),
    ("Testing Sample", len(testing_sample)),
    ("Prompt Experiment Rows", len(prompt_experiment)),
    ("Communication Styles", len(communication_prompts))
)

Notebook Complete
Output Folder
Data\Narratives
--------------------------------------------------------------------------------
Training Sample
5
--------------------------------------------------------------------------------
Development Sample
5
--------------------------------------------------------------------------------
Testing Sample
5
--------------------------------------------------------------------------------
Prompt Experiment Rows
60
--------------------------------------------------------------------------------
Communication Styles
4
--------------------------------------------------------------------------------
